# Creating a FabSim3 plugin from FabDummy

**Project:** SEAVEA IAA (RA training task) | **Plugin built here:** `FabTest` | **Example application:** `cannon_app`

FabSim3 automates running simulations locally or on HPC machines. A **plugin** is a small folder that teaches FabSim3 how to run *one specific application*. This notebook walks through creating your own plugin by starting from **FabDummy**, the minimal template plugin, and ends with a plugin that runs a real (small) Python application as a single job and as an ensemble.

Source documentation: <https://fabsim3.readthedocs.io/en/latest/create_new_plugin/>

**What you will do**
1. Check your FabSim3 install and run the stock FabDummy test.
2. Copy FabDummy into your own plugin, `FabTest`, and rename it.
3. Write two tasks (single job, ensemble), a job template and a machine config.
4. Add an application (`cannon_app`) and its ensemble inputs.
5. Register, install and run the plugin, then inspect the results.

**How to use this notebook:** run the cells top to bottom (*Kernel > Restart & Run All*). Cells are written so that re-running them is safe: existing files are kept or overwritten with the same content.

**Assumptions:** Windows 11 with **WSL2 Ubuntu** and Jupyter started *inside* WSL; FabSim3 installed in `~/FabSim3`; `ssh localhost` works without a password (see *Prerequisites*).

## Progress status (for the progress meeting)

| Section | Status |
| --- | --- |
| 1-3. Setup, prerequisites, stock FabDummy run | Done and tested on WSL2 (feedback log entries //1-//9) |
| 4-6. Create `FabTest` skeleton, tasks, template, config | Drafted, being tested |
| 7-8. `cannon_app`, ensemble inputs, `requirements.txt` | Drafted, being tested |
| 9-11. Register, install, run, inspect results | Drafted, being tested |
| 12. Troubleshooting table | Based on issues already logged (//5, //7, //12, //18, //19) |

Cells that have not yet been run end to end in the target environment are marked **(untested)**.

## 1. Background: what a plugin looks like

A plugin lives in `FabSim3/plugins/<PluginName>/` and has this layout:

```
FabTest/
├── FabTest.py                    # the tasks (what you can type after "fabsim localhost ...")
├── machines_FabTest_user.yml     # plugin-specific settings per machine (optional but useful)
├── requirements.txt              # extra Python packages the plugin needs (optional)
├── config_files/                 # one sub-folder per "app": the inputs/code that get copied to the job
│   └── cannon_app/
│       ├── cannonsim.py
│       ├── input_files/simsetting.txt
│       └── SWEEP/                # ensemble variants (one folder per run)
└── templates/
    └── fabtest                   # the job script template that FabSim3 fills in and runs
```

**Three names must be identical:** the plugin name (`FabTest`), the Python file (`FabTest.py`) and the entry in `FabSim3/fabsim/deploy/plugins.yml`. The folder names `config_files` and `templates` are fixed by FabSim3.

**How a task runs:** `fabsim localhost fabtest:cannon_app` calls the task `fabtest` with `app="cannon_app"`. FabSim3 copies `config_files/cannon_app` to the target machine, fills in `templates/fabtest` with variables from the machine config files, and runs the resulting script there.

**FabDummy** is the smallest working example of this layout, which is why we copy it rather than start from nothing.

## 2. Setup and prerequisites

The next cell defines paths and a small helper, `sh()`, that runs a shell command and prints its output together with the exit code. Using one helper means every command runs with the same `PATH`, even if Jupyter was not started from a shell that has the FabSim3 virtual environment active.

In [ ]:
import os
import re
import shutil
import subprocess
from pathlib import Path

HOME = Path.home()
FABSIM = HOME / "FabSim3"                  # FabSim3 install
PLUGIN = "FabTest"                         # our plugin name
SRC = HOME / PLUGIN                        # our plugin's own source folder (kept outside FabSim3)
INSTALLED = FABSIM / "plugins" / PLUGIN    # where install_plugin puts the copy FabSim3 actually uses
APP = "cannon_app"                         # the application we wrap

# Make `fabsim` and the FabSim3 virtual environment visible to every command run from this notebook
extra_path = [str(FABSIM / "VirtualEnv" / "bin"), str(FABSIM / "fabsim" / "bin"), str(HOME / ".local" / "bin")]
os.environ["PATH"] = os.pathsep.join(extra_path + [os.environ.get("PATH", "")])
os.environ["PYTHONPATH"] = os.pathsep.join(
    p for p in [str(FABSIM), str(HOME / "flee"), os.environ.get("PYTHONPATH", "")] if p
)
os.environ["SRC"] = str(SRC)


def sh(cmd, timeout=900):
    """Run a bash command, print combined stdout/stderr, and return the exit code."""
    result = subprocess.run(["bash", "-c", cmd], capture_output=True, text=True, timeout=timeout)
    output = (result.stdout + result.stderr).rstrip()
    if output:
        print(output)
    print(f"[exit code {result.returncode}]")
    return result.returncode


print("FabSim3 folder :", FABSIM, "(found)" if FABSIM.exists() else "(NOT FOUND - install FabSim3 first)")
print("Plugin source  :", SRC)

### Prerequisite check

All three checks must pass before continuing:

- `which fabsim` prints a path inside `~/FabSim3/` (otherwise the PATH exports are missing: feedback log //5).
- `python3 --version` prints a version.
- `ssh localhost` works without a password. FabSim3 runs even *localhost* jobs over SSH. On a fresh WSL install start the server with `sudo service ssh start` and set up an SSH key for localhost (feedback log //7). SSH must be restarted after each WSL restart.

In [ ]:
sh("which fabsim; python3 --version; ssh -o BatchMode=yes -o ConnectTimeout=5 localhost 'echo ssh localhost OK' 2>&1 | grep -v -i xauth")

## 3. Look at FabDummy and run its test

FabDummy ships with FabSim3 in `~/FabSim3/plugins/FabDummy`. First look at its layout: it is the template we are going to copy.

In [ ]:
sh(f"ls -R {FABSIM}/plugins/FabDummy | grep -v '^$' | head -40")

In [ ]:
# The tasks FabDummy defines (everything after "@task" is something you can run from the command line)
sh(f"grep -n -E '^@task|^def |add_local_paths|env.script' {FABSIM}/plugins/FabDummy/FabDummy.py")

### Baseline: run the stock FabDummy test

This proves FabSim3, SSH and the plugin system work *before* we change anything. If this fails, fix it first (see the troubleshooting table at the end). Expected final message: "This dummy test has been concluded with great success".

In [ ]:
sh("fabsim localhost dummy:dummy_test 2>&1 | grep -v -i -E 'xauth|X11'")
sh("fabsim localhost fetch_results 2>&1 | tail -5")

## 4. Create the FabTest plugin skeleton

### 4.1 Copy FabDummy

We copy FabDummy to a folder of our own, outside `~/FabSim3/plugins`. Keeping the source separate lets us version-control it and then *install* it the same way any other plugin is installed. If the folder already exists (for example you cloned it earlier), it is left alone.

In [ ]:
if SRC.exists():
    print(f"{SRC} already exists - keeping it")
else:
    shutil.copytree(FABSIM / "plugins" / "FabDummy", SRC, ignore=shutil.ignore_patterns(".git"))
    print("Copied FabDummy ->", SRC)

### 4.2 Rename the main file and the plugin name inside it

Two changes from the docs: rename `FabDummy.py` to `FabTest.py`, and change the argument of `add_local_paths("FabDummy")` to `add_local_paths("FabTest")`. `add_local_paths` tells FabSim3 to search this plugin's `templates/` and `config_files/` folders.

In [ ]:
old_py, new_py = SRC / "FabDummy.py", SRC / f"{PLUGIN}.py"
if old_py.exists() and not new_py.exists():
    old_py.rename(new_py)
    print(f"Renamed {old_py.name} -> {new_py.name}")
if new_py.exists():
    new_py.write_text(new_py.read_text().replace("FabDummy", PLUGIN))
sh(f"grep -n 'add_local_paths' {new_py}")

### 4.3 Remove FabDummy's own example content

FabDummy is already installed under its own name, and FabSim3 loads *every* installed plugin. If two plugins define a task, template or config folder with the same name (`dummy`), they can clash. So we delete the dummy-named template and test config from our copy; we replace them with our own below.

In [ ]:
for leftover in [SRC / "templates" / "dummy", SRC / "config_files" / "dummy_test"]:
    if leftover.is_dir():
        shutil.rmtree(leftover)
        print("removed", leftover.relative_to(SRC))
    elif leftover.exists():
        leftover.unlink()
        print("removed", leftover.relative_to(SRC))
sh(f"cd {SRC} && ls -R . | grep -v '^$'")

## 5. Write the tasks

A **task** is a Python function tagged with `@task`. We write two, following the pattern in the docs:

| Task | Command | What it does |
| --- | --- | --- |
| `fabtest` | `fabsim localhost fabtest:cannon_app` | runs **one** job |
| `fabtest_ensemble` | `fabsim localhost fabtest_ensemble:cannon_app` | runs **one job per folder** in `cannon_app/SWEEP` |

What each line means:

| Line | Meaning |
| --- | --- |
| `from fabsim.base.fab import *` | loads FabSim3's built-in functions |
| `add_local_paths("FabTest")` | register this plugin's `templates/` and `config_files/` folders |
| `@task` | makes the function callable from the `fabsim` command line |
| `@load_plugin_env_vars("FabTest")` | loads settings from `machines_FabTest_user.yml` |
| `update_environment(args)` | merges command-line arguments into FabSim3's environment |
| `with_config(app)` | selects the config folder `config_files/<app>` |
| `execute(put_configs, app)` | copies that config folder to the (local or remote) machine |
| `env.script = "fabtest"` | name of the template file in `templates/` to use for the job |
| `job(args)` | submits the job |
| `find_config_file_path(app) + "/SWEEP"` | path of the folder holding the ensemble variants |
| `run_ensemble(app, sweep_dir, **args)` | creates and submits one job per `SWEEP` sub-folder (it also copies the configs, so no `put_configs` call is needed) |

In [ ]:
TASKS = """from fabsim.base.fab import *

# Add this plugin's templates/ and config_files/ folders to FabSim3's search path
add_local_paths("FabTest")


@task
@load_plugin_env_vars("FabTest")
def fabtest(app, **args):
    # Submit a single job of an application stored in config_files/<app>
    #   fabsim localhost fabtest:cannon_app
    update_environment(args)
    with_config(app)
    execute(put_configs, app)
    env.script = "fabtest"
    job(args)


@task
@load_plugin_env_vars("FabTest")
def fabtest_ensemble(app, **args):
    # Submit an ensemble: one job per sub-folder of config_files/<app>/SWEEP
    #   fabsim localhost fabtest_ensemble:cannon_app
    update_environment(args)
    with_config(app)
    sweep_dir = find_config_file_path(app) + "/SWEEP"
    env.script = "fabtest"
    run_ensemble(app, sweep_dir, **args)
"""

(SRC / f"{PLUGIN}.py").write_text(TASKS)
sh(f"cat {SRC}/{PLUGIN}.py")

## 6. Job template and machine configuration

### 6.1 The job template

`templates/fabtest` is the script FabSim3 runs for each job. FabSim3 uses Python **template substitution**: `$name` is replaced by the value of the variable `name`, and `$$` becomes a literal `$`.

- `$job_results` : the folder the job runs in (FabSim3 provides it)
- `$run_prefix` : an optional machine-specific prefix (FabSim3 provides it)
- `$py_app_run_command` : defined by *us* in the machine config below

In [ ]:
TEMPLATE = """# change directory to where the job was copied
cd $job_results
$run_prefix

# run the application (the command comes from machines_FabTest_user.yml)
$py_app_run_command

echo "Output results:"
cat output_files/py_output.txt
"""

(SRC / "templates").mkdir(exist_ok=True)
(SRC / "templates" / "fabtest").write_text(TEMPLATE)
sh(f"cat {SRC}/templates/fabtest")

### 6.2 The plugin's machine configuration

`@load_plugin_env_vars("FabTest")` reads `machines_FabTest_user.yml` from the plugin folder. The top-level key (`default`) applies to every machine; you could add a key named after a machine (e.g. `localhost`) to override values per machine. FabSim3 merges settings in this order: `machines.yml`, then `machines_user.yml`, then this plugin file, then command-line arguments.

We define one variable, the command that runs our application. We use `python3` because FabSim3 jobs start via SSH and resolve `python3` to the **system** Python, not the FabSim3 virtual environment (feedback log //18).

In [ ]:
MACHINE_CFG = """default:
  py_app_run_command: "python3 cannonsim.py"
"""

(SRC / f"machines_{PLUGIN}_user.yml").write_text(MACHINE_CFG)
sh(f"cat {SRC}/machines_{PLUGIN}_user.yml")

## 7. Add the application and ensemble inputs

### 7.1 The application: `cannon_app`

`cannon_app` (<https://github.com/arabnejad/cannon_app>) is a tiny physics simulation from the FabSim3 docs: it computes how far a projectile travels from its speed, angle, gravity and air resistance. It reads `input_files/simsetting.txt` and writes `output_files/py_output.txt` and a plot. We clone it into `config_files/` (requires internet access).

In [ ]:
CFG = SRC / "config_files"
CFG.mkdir(parents=True, exist_ok=True)

if not (CFG / APP).exists():
    sh(f"cd {CFG} && git clone https://github.com/arabnejad/cannon_app.git && rm -rf {APP}/.git")
sh(f"ls -R {CFG / APP} | grep -v '^$' | head -30")

The application's input file. We will vary one parameter in the ensemble.

In [ ]:
sim_file = CFG / APP / "input_files" / "simsetting.txt"
print(sim_file.read_text() if sim_file.exists() else "simsetting.txt not found - check the clone above")

### 7.2 Ensemble inputs (`SWEEP`)

An ensemble is a set of runs that differ in their inputs. For `run_ensemble`, create a `SWEEP` folder in the application folder with **one sub-folder per run**. Inside each, put the files that should differ, using the **same relative path** as the original (here `input_files/simsetting.txt`). FabSim3 copies the base application to each run and overlays the files from its `SWEEP/<run>` folder.

We make three runs that differ only in the launch `angle` (radians).

In [ ]:
def with_angle(text, new_angle):
    """Return the simsetting text with the 'angle' value replaced; second item says whether it was found."""
    lines, found = [], False
    for line in text.splitlines():
        first = line.split(",")[0].strip().strip('"').lower()
        if first == "angle":
            line, found = f'"angle",{new_angle}', True
        lines.append(line)
    return "\n".join(lines) + "\n", found


if sim_file.exists():
    base = sim_file.read_text()
    for run, angle in {"run_1": 0.3, "run_2": 0.5, "run_3": 0.7}.items():
        text, found = with_angle(base, angle)
        if not found:
            print("WARNING: no 'angle' line found in simsetting.txt - edit with_angle() to match its format")
        run_dir = CFG / APP / "SWEEP" / run / "input_files"
        run_dir.mkdir(parents=True, exist_ok=True)
        (run_dir / "simsetting.txt").write_text(text)
    sh(f"cd {CFG / APP} && find SWEEP -type f | sort")
else:
    print("Skipping: clone cannon_app first (section 7.1)")

## 8. `requirements.txt` (extra Python packages)

If your plugin needs Python packages, list them in a `requirements.txt` next to `FabTest.py`: one pip requirement per line, same format as any pip requirements file. FabSim3's `install_plugin` looks for this file (FabDummy has none and prints "No requirements.txt file found").

`cannon_app` plots its result with **matplotlib**, so that is our one requirement.

**Important:** `requirements.txt` installs packages for the *FabSim3 environment*. Our job, however, runs under the *system* `python3` (feedback log //18), so matplotlib must also be available there. The second cell checks and installs it if needed.

In [ ]:
(SRC / "requirements.txt").write_text("matplotlib\n")
sh(f"cat {SRC}/requirements.txt")

In [ ]:
# The job runs under the SYSTEM python3, so check/install matplotlib for /usr/bin/python3 specifically
sh("/usr/bin/python3 -c 'import matplotlib; print(\"matplotlib\", matplotlib.__version__, \"found for system python3\")' "
   "|| /usr/bin/python3 -m pip install matplotlib --break-system-packages")

## 9. Version-control the plugin and register it

### 9.1 Make it a git repository

`install_plugin` works by running `git clone <repository> FabSim3/plugins/<name>`. So our plugin folder needs to be a git repository with at least one commit (git only clones *committed* files). If the folder was cloned from FabDummy it still points at FabDummy's repository, so we reset it to a fresh repository of our own.

In [ ]:
sh("""cd "$SRC"
if git remote get-url origin 2>/dev/null | grep -q 'djgroen/FabDummy'; then rm -rf .git; echo "removed the old FabDummy git history"; fi
[ -d .git ] || git init -q
git config user.name  >/dev/null || git config user.name  "FabSim3 RA"
git config user.email >/dev/null || git config user.email "ra@example.com"
git add -A
git commit -q -m "FabTest plugin" || echo "nothing new to commit"
git log --oneline | head -3""")

### 9.2 Register the plugin in `plugins.yml` (untested)

FabSim3 finds installable plugins in `FabSim3/fabsim/deploy/plugins.yml`: one entry per plugin, with the name as key and an indented `repository:` line. For official plugins that is a GitHub URL; for ours we use the **local path** of our git repository, since `install_plugin` simply runs `git clone` on it. A GitHub URL (after pushing the plugin to your own repository) works the same way.

The cell backs the file up first (`plugins.yml.bak`) and only adds the entry if it is not already there.

In [ ]:
yml = FABSIM / "fabsim" / "deploy" / "plugins.yml"
text = yml.read_text()
if re.search(rf"^{PLUGIN}:", text, re.M):
    print(f"{PLUGIN} is already registered")
else:
    shutil.copy(yml, str(yml) + ".bak")
    if not text.endswith("\n"):
        text += "\n"
    yml.write_text(text + f"{PLUGIN}:\n    repository: {SRC}\n")
    print(f"Added {PLUGIN} (backup: plugins.yml.bak)")
sh(f"tail -6 {yml}")

## 10. Install the plugin (untested)

`install_plugin` clones our repository into `~/FabSim3/plugins/FabTest`, which is the copy FabSim3 actually uses. If you change the plugin later, commit the change in `~/FabTest` and run this cell again (FabSim3 keeps a backup of the previous copy).

Expect a message about the plugin being cloned. After install, FabSim3 loads the plugin on every command: any import or syntax error in `FabTest.py` will show up here.

In [ ]:
sh(f"fabsim localhost install_plugin:{PLUGIN} 2>&1 | grep -v -i -E 'xauth|X11'")
sh(f"ls {FABSIM}/plugins; echo; ls -R {INSTALLED} | grep -v '^$' | head -30")

## 11. Run the plugin (untested)

### 11.1 Single job

```
fabsim localhost fabtest:cannon_app
```

In [ ]:
sh(f"fabsim localhost fabtest:{APP} 2>&1 | grep -v -i -E 'xauth|X11'")

### 11.2 Ensemble (3 runs)

```
fabsim localhost fabtest_ensemble:cannon_app
```

In [ ]:
sh(f"fabsim localhost fabtest_ensemble:{APP} 2>&1 | grep -v -i -E 'xauth|X11'")

### 11.3 Fetch results

For `localhost` the jobs run in `~/FabSim3/results/`; `fetch_results` brings the output back into the usual results location.

In [ ]:
sh("fabsim localhost fetch_results 2>&1 | tail -8")

### 11.4 Check the results yourself

**Do not trust the "submission complete" message alone**: FabSim3 reports success even when every job crashed immediately (feedback log //19). The real check is whether `py_output.txt` exists and contains numbers.

In [ ]:
results = FABSIM / "results"
runs = sorted(results.glob(f"{APP}_localhost_*"), key=lambda p: p.stat().st_mtime, reverse=True) if results.exists() else []
print("Newest result folders:", [r.name for r in runs[:3]] or "none found")

outputs = [f for r in runs[:2] for f in r.rglob("py_output.txt")]
for f in outputs[:4]:
    print("\n==", f.relative_to(results))
    print(f.read_text())
if not outputs:
    print("No py_output.txt found -> the jobs probably failed. Open the newest result folder and read its error output.")

In [ ]:
from IPython.display import Image, display

plots = [p for r in runs[:2] for p in r.rglob("py_output.png")]
for p in plots[:2]:
    print(p.relative_to(results))
    display(Image(filename=str(p), width=420))
if not plots:
    print("No plot found yet.")

## 12. Troubleshooting

These are real problems met while setting up FabSim3 on WSL2 (the numbers refer to the SEAVEA feedback log).

| Symptom | Likely cause and fix |
| --- | --- |
| `fabsim: command not found` | PATH/PYTHONPATH exports missing; add them to `~/.bashrc` and activate the venv (//5) |
| `ssh: connect to host localhost port 22: Connection refused` | SSH server not running: `sudo service ssh start`; restart it after each WSL restart (//7) |
| Plugin import error on every command | A Python package the plugin needs is missing in the FabSim3 venv; install it there (//12) |
| `No module named 'pkg_resources'` | Python 3.14 venv with new setuptools: `pip install "setuptools<81"` (//13) |
| `The request task ... is not available` | Task name wrong, or plugin name, `.py` file name and `plugins.yml` key do not match; check spelling and that the plugin installed |
| `install_plugin` fails with a git error | Our plugin folder is not a git repo, has no commit, or the `repository:` path in `plugins.yml` is wrong (section 9) |
| `No module named ...` inside a job | The job runs under the system `python3`, not the venv; install the package for `/usr/bin/python3` (//18) |
| "Submission complete" but no output files | Jobs crashed silently (//19); read the error output in the newest `~/FabSim3/results/` folder |
| `NO machines_..._user.yml FOUND` | The machine config file is missing or misnamed; it must be `machines_<PluginName>_user.yml` in the plugin folder |
| `FileNotFoundError: input_files/simsetting.txt` | The app folder name passed to the task does not match `config_files/<app>` |
| Warnings about `xauth` / X11 | Harmless; ignored in this notebook (//8) |

## 13. Optional clean-up

Uncomment to remove the test plugin. This deletes the installed copy and your source folder; to unregister it, also restore `plugins.yml` from `plugins.yml.bak`.

In [ ]:
# shutil.rmtree(INSTALLED, ignore_errors=True)   # the installed copy FabSim3 uses
# shutil.rmtree(SRC, ignore_errors=True)         # your source folder
# shutil.copy(str(yml) + ".bak", yml)            # undo the plugins.yml entry
print("Nothing removed (clean-up lines are commented out).")

## Summary

You created a FabSim3 plugin by:

1. Copying **FabDummy** and renaming the plugin file and `add_local_paths(...)` name.
2. Writing `fabtest` and `fabtest_ensemble` **tasks**.
3. Adding a **job template** (`templates/fabtest`) and a **machine config** (`machines_FabTest_user.yml`).
4. Adding an application (`config_files/cannon_app`) with **ensemble inputs** (`SWEEP`).
5. Listing dependencies in `requirements.txt`.
6. Making the folder a git repository, **registering** it in `plugins.yml`, and **installing** it with `install_plugin`.

**Next steps:** run the plugin on a remote HPC machine (add the machine in `machines_user.yml`), add `replicas=N` to repeat each ensemble run, or connect the ensemble to **EasyVVUQ** for sensitivity analysis (see the FabFlee plugin for a worked example).

**References:** [Create new plugin](https://fabsim3.readthedocs.io/en/latest/create_new_plugin/) | [APIs for plugin developers](https://fabsim3.readthedocs.io/en/latest/APIs_for_plugin_developers/) | [FabDummy](https://github.com/djgroen/FabDummy) | [cannon_app](https://github.com/arabnejad/cannon_app)